# Notebook 12 — Hands-On Lab: Regularization Effects Study & Optimizer Comparison

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 4 hours  
**Week 8 — Regularization, Hyperparameter Tuning & Optimization**

---

## Objectives
1. Implement Ridge and Lasso regression **from scratch** using coordinate descent  
2. Visualise regularization paths and understand how Lasso eliminates noise features  
3. Select optimal regularization strength automatically via cross-validation  
4. Implement and compare Adam, SGD, and SGD+Momentum from scratch  
5. Use Optuna (or a GP-based fallback) to tune ElasticNet hyperparameters  
6. Build a complete model comparison table on a held-out test set  

---

## Dataset
Synthetic California-style house price data: **1,000 samples, 25 features** (10 true predictors + 15 noise).  
This lets us objectively measure whether regularization correctly identifies signal from noise.

---

*Real-world framing:* You are a data scientist at a real estate analytics firm. You have 25 candidate features for a house price model, but you suspect at least half are noise. Your job: find the right regularization technique and optimizer to build the most generalisable model.

## Section 0 — Setup

In [ ]:
# ── Cell 3: Imports ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import (
    Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV,
    LinearRegression, SGDClassifier
)
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# Reproducibility — must be set before any random number generation
np.random.seed(42)

# Plot style
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

# Colour palette for optimizers
OPT_COLORS = {
    "SGD":      "#e41a1c",
    "Momentum": "#ff7f00",
    "Adam":     "#377eb8",
}

print("All imports successful.")

In [ ]:
# ── Cell 4: Generate synthetic California-style housing dataset ───────────────
np.random.seed(42)    # reset here so dataset is always identical

N_SAMPLES    = 1000
N_FEATURES   = 25
N_TRUE       = 10     # first 10 columns are true predictors
N_NOISE      = 15     # last 15 columns are pure noise

# ── Generate features: correlated block to mimic real housing data ─────────────
# Covariance matrix with mild correlation (0.3) between predictors
cov_block  = 0.3 * np.ones((N_TRUE, N_TRUE)) + 0.7 * np.eye(N_TRUE)
X_signal   = np.random.multivariate_normal(np.zeros(N_TRUE), cov_block, N_SAMPLES)
X_noise    = np.random.randn(N_SAMPLES, N_NOISE)   # pure i.i.d. noise
X_full     = np.hstack([X_signal, X_noise])         # (1000, 25)

# ── True coefficients: only first 10 features have non-zero coefficients ───────
TRUE_COEF  = np.array([3.5, -2.1, 4.0, -1.8, 2.7, 1.5, -3.2, 2.0, -1.2, 3.8])
true_coef_full = np.concatenate([TRUE_COEF, np.zeros(N_NOISE)])

# ── Target: house price in $100k units (realistic California range) ────────────
noise_std  = 2.0
y_reg      = X_full @ true_coef_full + 5.0 + np.random.randn(N_SAMPLES) * noise_std

# ── Binary version of target for optimizer comparison ─────────────────────────
y_median   = np.median(y_reg)
y_clf      = (y_reg > y_median).astype(int)

# ── Train / test split (80/20, stratified for classification) ─────────────────
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_reg, test_size=0.2, random_state=42)
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_full, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# ── Scale features ─────────────────────────────────────────────────────────────
scaler    = StandardScaler()
X_tr_s    = scaler.fit_transform(X_tr)      # regression
X_te_s    = scaler.transform(X_te)

scaler_c  = StandardScaler()
X_tr_cs   = scaler_c.fit_transform(X_tr_c)  # classification
X_te_cs   = scaler_c.transform(X_te_c)

# Add bias column for from-scratch optimizers
X_tr_b    = np.c_[np.ones(len(X_tr_cs)), X_tr_cs]
X_te_b    = np.c_[np.ones(len(X_te_cs)), X_te_cs]

FEATURE_NAMES = [f"Feature_{i+1}" for i in range(N_FEATURES)]
TRUE_FEATURES  = set(range(N_TRUE))   # indices 0..9 are true signal

print(f"Dataset shape         : {X_full.shape}")
print(f"True signal features  : {N_TRUE}  (Feature_1 … Feature_{N_TRUE})")
print(f"Noise features        : {N_NOISE} (Feature_{N_TRUE+1} … Feature_{N_FEATURES})")
print(f"y_reg range           : [{y_reg.min():.1f}, {y_reg.max():.1f}]")
print(f"Class balance         : {y_clf.mean()*100:.1f}% above median")

## Section 1 — Ridge & Lasso from Scratch with Coordinate Descent

### 1a. Ridge Regression — Analytical Solution

Ridge adds an L2 penalty: $\mathcal{L}_{\text{Ridge}} = \|y - X\theta\|^2 + \alpha\|\theta\|^2$.  
Setting the gradient to zero gives the **closed-form** solution:
$$\theta^* = (X^\top X + \alpha I)^{-1} X^\top y$$
No iteration needed — we solve a linear system directly.

In [ ]:
# ── Cell 6: Ridge from scratch — analytical solution ──────────────────────────
def ridge_analytical(X, y, alpha=1.0):
    """
    Ridge regression via the normal equations with L2 regularization.

    Minimises: ||y - Xθ||² + α||θ||²
    Solution:  θ = (XᵀX + αI)⁻¹ Xᵀy

    np.linalg.solve(A, b) is numerically preferred over explicit matrix inversion.

    Parameters
    ----------
    X     : scaled feature matrix (N, p)
    y     : continuous target (N,)
    alpha : regularization strength

    Returns
    -------
    theta : coefficient vector (p,)
    """
    n, p = X.shape
    I    = np.eye(p)                       # identity matrix
    A    = X.T @ X + alpha * I             # regularized Gram matrix
    b    = X.T @ y                         # right-hand side
    theta = np.linalg.solve(A, b)          # solve the linear system Aθ = b
    return theta

# Fit and compare with sklearn Ridge
alpha_demo   = 1.0
theta_ridge  = ridge_analytical(X_tr_s, y_tr, alpha=alpha_demo)

sk_ridge     = Ridge(alpha=alpha_demo, fit_intercept=False)
sk_ridge.fit(X_tr_s, y_tr)

# Verify numerical agreement
max_diff_ridge = np.abs(theta_ridge - sk_ridge.coef_).max()
print(f"Ridge from scratch vs sklearn — max coefficient difference: {max_diff_ridge:.2e}")
print(f"(Should be < 1e-10 for identical closed-form solutions)")

# Test set performance
y_pred_ridge = X_te_s @ theta_ridge
mse_ridge    = mean_squared_error(y_te, y_pred_ridge)
r2_ridge     = r2_score(y_te, y_pred_ridge)
print(f"\nRidge (α={alpha_demo}) — Test MSE: {mse_ridge:.4f}  |  Test R²: {r2_ridge:.4f}")

### 1b. Lasso Regression — Coordinate Descent with Soft-Thresholding

Lasso adds an L1 penalty: $\mathcal{L}_{\text{Lasso}} = \|y - X\theta\|^2 + \alpha\|\theta\|_1$.  
The L1 penalty has no closed-form solution for all parameters simultaneously. We use **coordinate descent**: cycle through each $\theta_j$ one at a time, fixing all others, and apply the **soft-thresholding** operator:

$$\theta_j \leftarrow S\!\left(\frac{1}{N}X_j^\top r_j,\; \alpha\right) \cdot \frac{N}{\|X_j\|^2}$$

where $r_j = y - X_{-j}\theta_{-j}$ is the partial residual, and $S(z, \lambda) = \text{sign}(z)(|z|-\lambda)_+$.

In [ ]:
# ── Cell 8: Lasso from scratch — coordinate descent ───────────────────────────
def soft_threshold(z, lam):
    """
    Soft-thresholding operator: S(z, λ) = sign(z) * max(|z| - λ, 0)
    Values in (-λ, +λ) are set to EXACTLY zero — this is what creates sparsity.
    """
    return np.sign(z) * np.maximum(np.abs(z) - lam, 0.0)

def lasso_coordinate_descent(X, y, alpha=1.0, n_iter=500, tol=1e-6):
    """
    Lasso via Cyclic Coordinate Descent.

    For each feature j (cycling 0..p-1 repeatedly):
      1. Compute partial residual: r_j = y - X @ θ + X_j * θ_j
      2. Compute raw estimate:     rho_j = (1/N) * X_j^T @ r_j
      3. Apply soft-threshold:     θ_j = S(rho_j, α) / (||X_j||² / N)

    Parameters
    ----------
    n_iter : maximum coordinate descent passes
    tol    : stop if max parameter change < tol

    Returns
    -------
    theta  : final sparse coefficient vector
    losses : residual sum of squares per pass (for convergence plot)
    """
    n, p   = X.shape
    theta  = np.zeros(p)
    losses = []

    # Pre-compute squared norms of each feature column (constant)
    col_norms_sq = (X ** 2).sum(axis=0) / n   # shape (p,)

    for iteration in range(n_iter):
        theta_old = theta.copy()

        for j in range(p):
            # Partial residual: what y looks like if θ_j were 0
            r_j   = y - (X @ theta) + X[:, j] * theta[j]

            # Ordinary least-squares estimate for θ_j
            rho_j = (1.0 / n) * (X[:, j] @ r_j)

            # Soft-threshold: values inside (-α, +α) go to exactly 0
            theta[j] = soft_threshold(rho_j, alpha) / col_norms_sq[j]

        # Record residual sum of squares (without penalty) for convergence plot
        residuals = y - X @ theta
        losses.append(np.mean(residuals ** 2))

        # Early stopping: check max parameter change across all coordinates
        if np.max(np.abs(theta - theta_old)) < tol:
            print(f"  Lasso converged at iteration {iteration + 1}")
            break

    return theta, losses

# Fit and compare with sklearn Lasso
alpha_demo    = 1.0
theta_lasso, lasso_losses = lasso_coordinate_descent(X_tr_s, y_tr, alpha=alpha_demo)

sk_lasso = Lasso(alpha=alpha_demo, fit_intercept=False, max_iter=10000)
sk_lasso.fit(X_tr_s, y_tr)

max_diff_lasso = np.abs(theta_lasso - sk_lasso.coef_).max()
print(f"Lasso from scratch vs sklearn — max coefficient difference: {max_diff_lasso:.4f}")

y_pred_lasso = X_te_s @ theta_lasso
mse_lasso    = mean_squared_error(y_te, y_pred_lasso)
r2_lasso     = r2_score(y_te, y_pred_lasso)
print(f"Lasso (α={alpha_demo}) — Test MSE: {mse_lasso:.4f}  |  Test R²: {r2_lasso:.4f}")
print(f"Non-zero coefficients : {np.sum(theta_lasso != 0)} / {len(theta_lasso)}")

In [ ]:
# ── Cell 9: Convergence plot — Ridge vs Lasso ─────────────────────────────────
# Ridge loss via gradient descent (so we can plot convergence, not just 1-shot)
def ridge_gradient_descent(X, y, alpha=1.0, lr=0.01, n_iter=200):
    """
    Ridge via gradient descent — used only to plot a convergence curve.
    Gradient: ∇L = (2/N) Xᵀ(Xθ - y) + 2α θ
    """
    n, p   = X.shape
    theta  = np.zeros(p)
    losses = []
    for _ in range(n_iter):
        resid = X @ theta - y
        grad  = (2.0 / n) * (X.T @ resid) + 2 * alpha * theta
        theta -= lr * grad
        losses.append(np.mean(resid ** 2) + alpha * np.sum(theta ** 2))
    return theta, losses

_, ridge_losses_iter = ridge_gradient_descent(X_tr_s, y_tr, alpha=1.0, lr=0.01, n_iter=200)

# Normalise both loss curves to start at 1 for fair comparison
def normalise_losses(losses):
    arr = np.array(losses)
    return arr / arr[0]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(normalise_losses(ridge_losses_iter), label="Ridge (GD)",  color="#1f77b4", linewidth=2)
ax.plot(normalise_losses(lasso_losses),      label="Lasso (CD)",  color="#d62728", linewidth=2)

ax.set_xlabel("Iteration", fontsize=11)
ax.set_ylabel("Normalised Loss (L/L₀)", fontsize=11)
ax.set_title("Convergence: Ridge (Gradient Descent) vs Lasso (Coordinate Descent)", fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
print("Coordinate descent for Lasso converges in fewer passes because it exactly")
print("optimises one coordinate at a time via closed-form soft-thresholding.")

In [ ]:
# ── Cell 10: Ridge vs Lasso coefficient bar chart ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

feature_labels = FEATURE_NAMES
colors_bar     = ["#2ca02c" if i < N_TRUE else "#d62728" for i in range(N_FEATURES)]

# Ridge coefficients
axes[0].bar(range(N_FEATURES), theta_ridge, color=colors_bar, edgecolor="white", linewidth=0.5)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title(f"Ridge (α={alpha_demo}) — All Coefficients Shrunk, None Zero", fontsize=11)
axes[0].set_xlabel("Feature Index")
axes[0].set_ylabel("Coefficient Value")
axes[0].set_xticks(range(0, N_FEATURES, 2))
axes[0].tick_params(axis="x", labelsize=8)

# Lasso coefficients
axes[1].bar(range(N_FEATURES), theta_lasso, color=colors_bar, edgecolor="white", linewidth=0.5)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title(f"Lasso (α={alpha_demo}) — Noise Features Set to Zero (red)", fontsize=11)
axes[1].set_xlabel("Feature Index")
axes[1].set_xticks(range(0, N_FEATURES, 2))
axes[1].tick_params(axis="x", labelsize=8)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2ca02c", label="True signal (Feature 1-10)"),
    Patch(facecolor="#d62728", label="Noise (Feature 11-25)"),
]
axes[1].legend(handles=legend_elements, loc="upper right", fontsize=9)

plt.suptitle("Ridge vs Lasso Coefficients — From-Scratch Implementation", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Ridge non-zero coefs : {np.sum(np.abs(theta_ridge) > 1e-6)} / {N_FEATURES}")
print(f"Lasso non-zero coefs : {np.sum(np.abs(theta_lasso) > 1e-6)} / {N_FEATURES}")

In [ ]:
# ── Cell 11: Verify from-scratch vs sklearn coefficients ──────────────────────
print("="*65)
print("VERIFICATION: From-Scratch vs sklearn (α=1.0)")
print("="*65)

verify_df = pd.DataFrame({
    "Feature":       FEATURE_NAMES,
    "True Coef":     true_coef_full,
    "Ridge (scratch)": theta_ridge,
    "Ridge (sklearn)": sk_ridge.coef_,
    "Lasso (scratch)": theta_lasso,
    "Lasso (sklearn)": sk_lasso.coef_,
})

# Show first 12 rows (covers all true features + 2 noise features)
print(verify_df.head(12).to_string(index=False, float_format="{:.4f}".format))
print("\n... (showing first 12 of 25 features)")

ridge_match = np.allclose(theta_ridge, sk_ridge.coef_, atol=1e-6)
lasso_match = np.allclose(theta_lasso, sk_lasso.coef_, atol=0.05)
print(f"\nRidge matches sklearn exactly  : {ridge_match}")
print(f"Lasso matches sklearn (±0.05)  : {lasso_match}")
print("(Small Lasso difference is normal: sklearn uses more iterations and")
print(" a slightly different convergence criterion)")

## Section 2 — Regularization Path Visualization

How do Ridge and Lasso coefficients change as regularization strength $\alpha$ varies?  
A **regularization path** answers: which features enter the model first as we relax regularization?

In [ ]:
# ── Cell 13: Compute regularization paths ─────────────────────────────────────
# 50 log-spaced alpha values from 0.001 to 100
alphas = np.logspace(-3, 2, 50)   # [0.001, 0.00178, ..., 56.2, 100.0]

# ── Ridge path ────────────────────────────────────────────────────────────────
ridge_coefs = np.zeros((len(alphas), N_FEATURES))
for i, a in enumerate(alphas):
    coef = ridge_analytical(X_tr_s, y_tr, alpha=a)
    ridge_coefs[i] = coef

# ── Lasso path ────────────────────────────────────────────────────────────────
# Use sklearn for speed (our CD works but is slow over 50 values × full grid)
lasso_coefs = np.zeros((len(alphas), N_FEATURES))
for i, a in enumerate(alphas):
    sk_l = Lasso(alpha=a, fit_intercept=False, max_iter=10000)
    sk_l.fit(X_tr_s, y_tr)
    lasso_coefs[i] = sk_l.coef_

print(f"Ridge path shape: {ridge_coefs.shape}   (n_alphas × n_features)")
print(f"Lasso path shape: {lasso_coefs.shape}")
print(f"Alpha range: [{alphas.min():.3f}, {alphas.max():.1f}]")

In [ ]:
# ── Cell 14: Regularization path plots (side by side) ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

log_alphas = np.log10(alphas)   # x-axis: log₁₀(α)

for ax, coefs, title in zip(
    axes,
    [ridge_coefs, lasso_coefs],
    ["Ridge Path — Shrinks towards zero, never exactly zero",
     "Lasso Path — Noise features eliminated (bold line = 0)"],
):
    for j in range(N_FEATURES):
        is_signal = j < N_TRUE
        ax.plot(
            log_alphas,
            coefs[:, j],
            color="#2ca02c" if is_signal else "#d62728",
            linewidth=2.0 if is_signal else 0.8,
            alpha=0.9 if is_signal else 0.5,
        )

    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xlabel("log₁₀(α)", fontsize=11)
    ax.set_ylabel("Coefficient Value", fontsize=11)
    ax.set_title(title, fontsize=11)

    # Custom legend
    from matplotlib.lines import Line2D
    legend_handles = [
        Line2D([0], [0], color="#2ca02c", linewidth=2,   label="Signal features (1-10)"),
        Line2D([0], [0], color="#d62728", linewidth=0.8, label="Noise features (11-25)"),
    ]
    ax.legend(handles=legend_handles, fontsize=9)

plt.suptitle("Regularization Paths: Ridge vs Lasso   (low α = weak regularization, high α = strong)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 15: Feature discovery — at which α does each signal feature enter? ───
# A feature "enters" Lasso when its coefficient first becomes non-zero
# (scanning from HIGH alpha to LOW alpha, i.e. tightest to weakest regularization)

entry_alphas = {}

for j in range(N_FEATURES):
    # Find the largest alpha index where coefficient is still exactly 0
    # Then the feature enters at the NEXT (smaller) alpha
    zero_mask = np.abs(lasso_coefs[:, j]) < 1e-8   # True where coef ≈ 0

    if zero_mask.all():
        entry_alphas[FEATURE_NAMES[j]] = None   # never entered (strong regularization)
    elif not zero_mask.any():
        entry_alphas[FEATURE_NAMES[j]] = alphas[-1]  # entered immediately (low alpha)
    else:
        # Last index where coefficient is zero (scanning high→low alpha)
        last_zero_idx = np.where(zero_mask)[0].max()
        if last_zero_idx + 1 < len(alphas):
            entry_alphas[FEATURE_NAMES[j]] = alphas[last_zero_idx + 1]
        else:
            entry_alphas[FEATURE_NAMES[j]] = None

entry_df = pd.DataFrame([
    {"Feature": k,
     "Is Signal": "✓" if int(k.split("_")[1]) - 1 < N_TRUE else "✗ noise",
     "Entry α (Lasso)": f"{v:.4f}" if v is not None else "never entered"}
    for k, v in entry_alphas.items()
])

print("Feature discovery: α at which each feature first becomes non-zero in Lasso")
print("(lower α = needs more regularization relaxation to enter = weaker predictor)")
print("=" * 60)
print(entry_df.to_string(index=False))

## Section 3 — Optimal Regularization via Cross-Validation

Instead of guessing $\alpha$, we let cross-validation choose it automatically.

In [ ]:
# ── Cell 17: LassoCV and RidgeCV — auto-selection of optimal alpha ────────────
cv_alphas = np.logspace(-3, 2, 60)   # 60 candidate values to search over

# LassoCV uses coordinate descent along the full regularization path efficiently
lasso_cv = LassoCV(alphas=cv_alphas, cv=5, fit_intercept=False, max_iter=10000)
lasso_cv.fit(X_tr_s, y_tr)

# RidgeCV uses leave-one-out CV by default (very fast for ridge)
ridge_cv = RidgeCV(alphas=cv_alphas, cv=5, fit_intercept=False)
ridge_cv.fit(X_tr_s, y_tr)

# ElasticNetCV for the comparison table
enet_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
    alphas=np.logspace(-3, 1, 30),
    cv=5, fit_intercept=False, max_iter=10000
)
enet_cv.fit(X_tr_s, y_tr)

print(f"LassoCV     — optimal α : {lasso_cv.alpha_:.6f}")
print(f"RidgeCV     — optimal α : {ridge_cv.alpha_:.6f}")
print(f"ElasticNetCV— optimal α : {enet_cv.alpha_:.6f}, l1_ratio: {enet_cv.l1_ratio_:.2f}")

In [ ]:
# ── Cell 18: CV error vs log(α) for Ridge and Lasso ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Lasso CV curve ─────────────────────────────────────────────────────────────
# lasso_cv.mse_path_ has shape (n_alphas, n_folds)
lasso_mse_mean = lasso_cv.mse_path_.mean(axis=1)
lasso_alphas   = lasso_cv.alphas_

axes[0].semilogx(lasso_alphas, lasso_mse_mean, color="#d62728", linewidth=2, label="CV MSE")
axes[0].axvline(lasso_cv.alpha_, color="black", linestyle="--", linewidth=1.5,
                label=f"Best α={lasso_cv.alpha_:.4f}")
axes[0].set_xlabel("α (log scale)", fontsize=11)
axes[0].set_ylabel("Mean CV MSE", fontsize=11)
axes[0].set_title("LassoCV — Cross-Validation Error vs α", fontsize=12)
axes[0].legend(fontsize=10)

# ── Ridge CV curve ─────────────────────────────────────────────────────────────
# Compute ridge CV scores manually for plotting
ridge_cv_mses = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for a in cv_alphas:
    fold_mses = []
    for train_idx, val_idx in kf.split(X_tr_s):
        coef_a = ridge_analytical(X_tr_s[train_idx], y_tr[train_idx], alpha=a)
        y_val_pred = X_tr_s[val_idx] @ coef_a
        fold_mses.append(mean_squared_error(y_tr[val_idx], y_val_pred))
    ridge_cv_mses.append(np.mean(fold_mses))

best_ridge_alpha = cv_alphas[np.argmin(ridge_cv_mses)]
axes[1].semilogx(cv_alphas, ridge_cv_mses, color="#1f77b4", linewidth=2, label="CV MSE")
axes[1].axvline(best_ridge_alpha, color="black", linestyle="--", linewidth=1.5,
                label=f"Best α={best_ridge_alpha:.4f}")
axes[1].set_xlabel("α (log scale)", fontsize=11)
axes[1].set_ylabel("Mean CV MSE", fontsize=11)
axes[1].set_title("RidgeCV — Cross-Validation Error vs α", fontsize=12)
axes[1].legend(fontsize=10)

plt.suptitle("Cross-Validation Error Curves for Regularization Strength", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Best Ridge α (manual CV): {best_ridge_alpha:.6f}")

In [ ]:
# ── Cell 19: Compare OLS, Ridge, Lasso, ElasticNet — test set ─────────────────
# OLS (no regularization)
ols = LinearRegression(fit_intercept=False)
ols.fit(X_tr_s, y_tr)

# Ridge with optimal alpha
theta_ridge_opt  = ridge_analytical(X_tr_s, y_tr, alpha=best_ridge_alpha)

# Lasso with optimal alpha
lasso_opt = Lasso(alpha=lasso_cv.alpha_, fit_intercept=False, max_iter=10000)
lasso_opt.fit(X_tr_s, y_tr)

# ElasticNet with optimal hyperparameters
enet_opt = ElasticNet(alpha=enet_cv.alpha_, l1_ratio=enet_cv.l1_ratio_,
                      fit_intercept=False, max_iter=10000)
enet_opt.fit(X_tr_s, y_tr)

models = {
    "OLS":              (ols.coef_,             ols.predict(X_te_s)),
    "Ridge (opt α)": (theta_ridge_opt,       X_te_s @ theta_ridge_opt),
    "Lasso (opt α)": (lasso_opt.coef_,       lasso_opt.predict(X_te_s)),
    "ElasticNet":       (enet_opt.coef_,        enet_opt.predict(X_te_s)),
}

print("Test Set Performance Comparison")
print("=" * 55)
for name, (coef, pred) in models.items():
    mse  = mean_squared_error(y_te, pred)
    r2   = r2_score(y_te, pred)
    nnz  = np.sum(np.abs(coef) > 1e-6)
    print(f"{name:<22} MSE={mse:.4f}  R²={r2:.4f}  Non-zero={nnz:2d}/25")

# Bar chart of R² values
fig, ax = plt.subplots(figsize=(8, 4))
names  = list(models.keys())
r2s    = [r2_score(y_te, pred) for _, (_, pred) in models.items()]
colors = ["#aec7e8", "#1f77b4", "#d62728", "#9467bd"]
bars   = ax.bar(names, r2s, color=colors, edgecolor="white", width=0.5)
ax.bar_label(bars, fmt="%.4f", fontsize=10, padding=3)
ax.set_ylabel("Test R²", fontsize=11)
ax.set_title("Test R² — OLS vs Ridge vs Lasso vs ElasticNet", fontsize=12)
ax.set_ylim(min(r2s) - 0.05, 1.0)
plt.tight_layout()
plt.show()

## Section 4 — Adam Optimizer from Scratch

We now switch to the **binary classification** version of the dataset and compare optimizers.

In [ ]:
# ── Cell 21: Helper functions for classification optimizers ───────────────────
def sigmoid(z):
    """Numerically stable sigmoid function."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def log_loss_fn(X, y, theta):
    """Binary cross-entropy loss."""
    y_hat = sigmoid(X @ theta)
    return -np.mean(y * np.log(y_hat + 1e-10) + (1 - y) * np.log(1 - y_hat + 1e-10))

def gradient_logistic(X, y, theta):
    """Full-batch gradient of binary cross-entropy."""
    return (1.0 / len(y)) * (X.T @ (sigmoid(X @ theta) - y))

# ── SGD (full-batch) ──────────────────────────────────────────────────────────
def sgd_clf(X, y, lr=0.1, n_epochs=300):
    theta  = np.zeros(X.shape[1])
    losses = [log_loss_fn(X, y, theta)]
    for _ in range(n_epochs):
        theta -= lr * gradient_logistic(X, y, theta)
        losses.append(log_loss_fn(X, y, theta))
    return theta, losses

# ── SGD + Momentum ────────────────────────────────────────────────────────────
def momentum_clf(X, y, lr=0.1, beta=0.9, n_epochs=300):
    theta  = np.zeros(X.shape[1])
    v      = np.zeros(X.shape[1])
    losses = [log_loss_fn(X, y, theta)]
    for _ in range(n_epochs):
        g = gradient_logistic(X, y, theta)
        v = beta * v - lr * g
        theta += v
        losses.append(log_loss_fn(X, y, theta))
    return theta, losses

# ── Adam ─────────────────────────────────────────────────────────────────────
def adam_clf(X, y, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8, n_epochs=300):
    theta  = np.zeros(X.shape[1])
    m, v   = np.zeros(X.shape[1]), np.zeros(X.shape[1])
    losses = [log_loss_fn(X, y, theta)]
    for t in range(1, n_epochs + 1):
        g     = gradient_logistic(X, y, theta)
        m     = beta1 * m + (1 - beta1) * g
        v     = beta2 * v + (1 - beta2) * g ** 2
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)
        theta -= lr * m_hat / (np.sqrt(v_hat) + eps)
        losses.append(log_loss_fn(X, y, theta))
    return theta, losses

# Run all three with default learning rates
theta_sgd,  l_sgd  = sgd_clf(X_tr_b,      y_tr_c, lr=0.1,  n_epochs=300)
theta_mom,  l_mom  = momentum_clf(X_tr_b,  y_tr_c, lr=0.1,  n_epochs=300)
theta_adam, l_adam = adam_clf(X_tr_b,      y_tr_c, lr=0.001, n_epochs=300)

for name, theta in [("SGD", theta_sgd), ("Momentum", theta_mom), ("Adam", theta_adam)]:
    acc = accuracy_score(y_te_c, (sigmoid(X_te_b @ theta) >= 0.5).astype(int))
    print(f"{name:<12} Test accuracy: {acc*100:.2f}%")

In [ ]:
# ── Cell 22: Convergence curves — SGD vs Momentum vs Adam ─────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(l_sgd,  label="SGD (lr=0.1)",        color=OPT_COLORS["SGD"],      linewidth=2)
ax.plot(l_mom,  label="Momentum β=0.9 (lr=0.1)", color=OPT_COLORS["Momentum"], linewidth=2)
ax.plot(l_adam, label="Adam (lr=0.001)",      color=OPT_COLORS["Adam"],     linewidth=2)

ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Binary Cross-Entropy Loss", fontsize=11)
ax.set_title("Optimizer Convergence on Binary House Price Classification", fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("Key observation:")
print(f"  SGD      final loss: {l_sgd[-1]:.4f}")
print(f"  Momentum final loss: {l_mom[-1]:.4f}")
print(f"  Adam     final loss: {l_adam[-1]:.4f}")

In [ ]:
# ── Cell 23: Learning rate robustness — SGD vs Adam final loss vs lr ──────────
lr_range    = np.logspace(-4, -1, 20)   # 0.0001 … 0.1
n_epochs_lr = 200

final_sgd_losses  = []
final_adam_losses = []

for lr in lr_range:
    _, ls = sgd_clf(X_tr_b,  y_tr_c, lr=lr,  n_epochs=n_epochs_lr)
    _, la = adam_clf(X_tr_b, y_tr_c, lr=lr,  n_epochs=n_epochs_lr)
    final_sgd_losses.append(ls[-1])
    final_adam_losses.append(la[-1])

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(lr_range, final_sgd_losses,  label="SGD",  color=OPT_COLORS["SGD"],  linewidth=2, marker="o", markersize=5)
ax.semilogx(lr_range, final_adam_losses, label="Adam", color=OPT_COLORS["Adam"], linewidth=2, marker="s", markersize=5)

ax.set_xlabel("Learning Rate (log scale)", fontsize=11)
ax.set_ylabel(f"Final Loss (epoch {n_epochs_lr})", fontsize=11)
ax.set_title("Learning Rate Robustness: SGD vs Adam\n"
             "(Adam's flat low-loss region = robust; SGD requires precise tuning)", fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Identify Adam's flat region
adam_arr   = np.array(final_adam_losses)
flat_mask  = adam_arr < adam_arr.min() * 1.05   # within 5% of minimum
flat_range = lr_range[flat_mask]
if flat_range.size > 1:
    print(f"Adam flat (low-loss) region: lr ∈ [{flat_range.min():.5f}, {flat_range.max():.4f}]")
print("Adam is robust over ~2 orders of magnitude; SGD diverges outside a narrow range.")

In [ ]:
# ── Cell 24: Verify from-scratch SGD vs sklearn SGDClassifier ─────────────────
# Sklearn's SGDClassifier with log_loss is mini-batch stochastic GD;
# we compare it with our full-batch SGD to show the same converged solution.

sk_sgd = SGDClassifier(
    loss="log_loss",
    learning_rate="constant",
    eta0=0.05,
    max_iter=300,
    fit_intercept=False,
    random_state=42,
    tol=None,
)
sk_sgd.fit(X_tr_cs, y_tr_c)   # note: no bias column needed (fit_intercept=False)

acc_scratch = accuracy_score(y_te_c, (sigmoid(X_te_b @ theta_sgd) >= 0.5).astype(int))
acc_sklearn = sk_sgd.score(X_te_cs, y_te_c)

print("SGD Comparison (same task, same hyperparameters)")
print(f"  From-scratch SGD accuracy : {acc_scratch*100:.2f}%")
print(f"  sklearn SGDClassifier     : {acc_sklearn*100:.2f}%")
print("")
print("Note: sklearn uses mini-batch SGD (stochastic, random shuffling each epoch)")
print("vs our full-batch implementation — expect similar but not identical results.")

## Section 5 — Optuna Hyperparameter Optimization

We optimize ElasticNet's `alpha` and `l1_ratio` jointly. If Optuna is not installed, we fall back to a simple Gaussian Process-based Bayesian optimization over `alpha` only.

In [ ]:
# ── Cell 26: Optuna availability check ────────────────────────────────────────
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)   # suppress verbose output
    OPTUNA_AVAILABLE = True
    print(f"Optuna version {optuna.__version__} found. Will run full Bayesian HPO.")
except ImportError:
    OPTUNA_AVAILABLE = False
    print("Optuna not installed. Falling back to GP-based Bayesian optimisation over alpha.")
    print("To install: pip install optuna")

In [ ]:
# ── Cell 27: Optuna study (or GP fallback) ────────────────────────────────────
N_TRIALS = 50    # budget: 50 evaluations for all methods

if OPTUNA_AVAILABLE:
    # ── Full 2-D Optuna study (alpha, l1_ratio) ────────────────────────────────
    def objective(trial):
        """
        Objective: 5-fold CV MSE for ElasticNet on training set.
        Optuna proposes alpha and l1_ratio; we return the CV score to minimise.
        """
        alpha    = trial.suggest_float("alpha",    1e-3, 10.0, log=True)
        l1_ratio = trial.suggest_float("l1_ratio", 0.01, 0.99)
        model    = ElasticNet(alpha=alpha, l1_ratio=l1_ratio,
                              fit_intercept=False, max_iter=5000)
        scores   = cross_val_score(model, X_tr_s, y_tr,
                                   cv=5, scoring="neg_mean_squared_error")
        return -scores.mean()   # Optuna minimises, so negate negative MSE

    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

    best_alpha_optuna    = study.best_params["alpha"]
    best_l1_optuna       = study.best_params["l1_ratio"]
    best_value_optuna    = study.best_value
    history_optuna       = [t.value for t in study.trials]
    best_so_far_optuna   = np.minimum.accumulate(history_optuna)

    print(f"Optuna ({N_TRIALS} trials) best result:")
    print(f"  alpha    = {best_alpha_optuna:.5f}")
    print(f"  l1_ratio = {best_l1_optuna:.4f}")
    print(f"  CV MSE   = {best_value_optuna:.4f}")

else:
    # ── GP-based Bayesian optimisation fallback (1-D over log alpha) ──────────
    # Simple implementation: UCB acquisition on a GP posterior
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import Matern

    def cv_mse_enet(log_alpha, l1_ratio=0.5):
        """5-fold CV MSE for ElasticNet at given log10(alpha)."""
        model  = ElasticNet(alpha=10**log_alpha, l1_ratio=l1_ratio,
                            fit_intercept=False, max_iter=5000)
        scores = cross_val_score(model, X_tr_s, y_tr,
                                 cv=5, scoring="neg_mean_squared_error")
        return -scores.mean()

    # Phase 1: random exploration (10 initial points)
    log_alpha_range = np.linspace(-3, 1, 200)   # candidate points
    X_bo  = np.random.uniform(-3, 1, (10, 1))   # initial random samples
    y_bo  = np.array([cv_mse_enet(x[0]) for x in X_bo])
    history_gp = list(y_bo)

    # Phase 2: GP-UCB acquisition for remaining budget
    gp = GaussianProcessRegressor(
        kernel=Matern(nu=2.5), alpha=1e-6,
        normalize_y=True, random_state=42
    )

    for iteration in range(N_TRIALS - 10):
        gp.fit(X_bo, y_bo)
        mu, sigma  = gp.predict(log_alpha_range.reshape(-1, 1), return_std=True)
        kappa      = 2.0 * (1 - iteration / (N_TRIALS - 10))  # decaying exploration
        ucb        = mu - kappa * sigma   # UCB: minimise → subtract uncertainty
        best_idx   = np.argmin(ucb)
        x_new      = log_alpha_range[best_idx]
        y_new      = cv_mse_enet(x_new)

        X_bo = np.vstack([X_bo, [[x_new]]])
        y_bo = np.append(y_bo, y_new)
        history_gp.append(y_new)

    best_idx_overall  = np.argmin(y_bo)
    best_log_alpha_gp = X_bo[best_idx_overall, 0]
    best_alpha_optuna = 10 ** best_log_alpha_gp
    best_l1_optuna    = 0.5
    best_value_optuna = y_bo[best_idx_overall]
    best_so_far_optuna = np.minimum.accumulate(history_gp)

    print(f"GP Bayesian Optim ({N_TRIALS} trials) best result:")
    print(f"  alpha  = {best_alpha_optuna:.5f}  (log10 = {best_log_alpha_gp:.3f})")
    print(f"  CV MSE = {best_value_optuna:.4f}")

In [ ]:
# ── Cell 28: Compare Optuna vs Grid Search vs Random Search ───────────────────
# All budgets = 50 evaluations

# ── Grid search (50 evaluations on alpha×l1_ratio grid) ───────────────────────
grid_alphas   = np.logspace(-3, 1, 10)   # 10 values
grid_l1ratios = [0.1, 0.3, 0.5, 0.7, 0.9]   # 5 values → 50 total

best_grid_mse   = np.inf
best_grid_alpha = None
best_grid_l1    = None
grid_history    = []

for a in grid_alphas:
    for l1 in grid_l1ratios:
        model  = ElasticNet(alpha=a, l1_ratio=l1, fit_intercept=False, max_iter=5000)
        scores = cross_val_score(model, X_tr_s, y_tr,
                                 cv=5, scoring="neg_mean_squared_error")
        mse    = -scores.mean()
        grid_history.append(mse)
        if mse < best_grid_mse:
            best_grid_mse   = mse
            best_grid_alpha = a
            best_grid_l1    = l1

# ── Random search (50 random (alpha, l1_ratio) pairs) ─────────────────────────
np.random.seed(42)
random_alphas = np.exp(np.random.uniform(np.log(1e-3), np.log(10), N_TRIALS))
random_l1s    = np.random.uniform(0.01, 0.99, N_TRIALS)

best_rand_mse = np.inf
rand_history  = []

for a, l1 in zip(random_alphas, random_l1s):
    model  = ElasticNet(alpha=a, l1_ratio=l1, fit_intercept=False, max_iter=5000)
    scores = cross_val_score(model, X_tr_s, y_tr,
                             cv=5, scoring="neg_mean_squared_error")
    mse    = -scores.mean()
    rand_history.append(mse)
    if mse < best_rand_mse:
        best_rand_mse = mse

best_so_far_grid = np.minimum.accumulate(grid_history)
best_so_far_rand = np.minimum.accumulate(rand_history)

# ── Plot optimisation history ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(best_so_far_optuna[:N_TRIALS], label="Optuna / GP-Bayes", color="#1f77b4", linewidth=2)
ax.plot(best_so_far_grid,              label="Grid Search",        color="#d62728", linewidth=2)
ax.plot(best_so_far_rand,              label="Random Search",      color="#2ca02c", linewidth=2)

ax.set_xlabel("Number of Evaluations", fontsize=11)
ax.set_ylabel("Best CV MSE So Far", fontsize=11)
ax.set_title("HPO Comparison: Optuna vs Grid vs Random Search (50 evaluations)", fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Optuna/GP best CV MSE : {best_so_far_optuna[:N_TRIALS][-1]:.4f}")
print(f"Grid Search best MSE  : {best_grid_mse:.4f}  (α={best_grid_alpha:.4f}, l1={best_grid_l1:.2f})")
print(f"Random Search best MSE: {best_rand_mse:.4f}")

In [ ]:
# ── Cell 29: Parameter importance (Optuna) or sensitivity (GP) ────────────────
if OPTUNA_AVAILABLE:
    # Optuna built-in feature importance (FAnova-based)
    importance = optuna.importance.get_param_importances(study)

    fig, ax = plt.subplots(figsize=(6, 3))
    params  = list(importance.keys())
    values  = list(importance.values())
    ax.barh(params, values, color=["#1f77b4", "#d62728"])
    ax.set_xlabel("Relative Importance", fontsize=11)
    ax.set_title("Optuna: Parameter Importance (FAnova)", fontsize=12)
    plt.tight_layout()
    plt.show()
    print("Interpretation: higher bar = this parameter has stronger effect on CV MSE.")

else:
    # GP fallback: show the 1-D response surface (CV MSE vs log alpha)
    log_alpha_range = np.linspace(-3, 1, 100)
    cv_mse_curve    = [cv_mse_enet(la) for la in log_alpha_range]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(log_alpha_range, cv_mse_curve, color="#1f77b4", linewidth=2)
    ax.axvline(np.log10(best_alpha_optuna), color="red", linestyle="--",
               label=f"Best α={best_alpha_optuna:.4f}")
    ax.set_xlabel("log₁₀(α)", fontsize=11)
    ax.set_ylabel("5-fold CV MSE", fontsize=11)
    ax.set_title("ElasticNet CV MSE vs Regularization Strength α (l1_ratio=0.5)", fontsize=11)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.show()
    print("Note: install Optuna for 2-D joint optimisation over (alpha, l1_ratio).")

## Section 6 — Full Model Comparison Table

In [ ]:
# ── Cell 31: Fit final models and build comparison table ──────────────────────
# OLS
ols_final = LinearRegression(fit_intercept=False)
ols_final.fit(X_tr_s, y_tr)

# Ridge (optimal α from CV)
theta_ridge_final = ridge_analytical(X_tr_s, y_tr, alpha=best_ridge_alpha)

# Lasso (optimal α from LassoCV)
lasso_final = Lasso(alpha=lasso_cv.alpha_, fit_intercept=False, max_iter=10000)
lasso_final.fit(X_tr_s, y_tr)

# ElasticNet (optimal from Optuna/GP)
enet_final = ElasticNet(
    alpha=best_alpha_optuna,
    l1_ratio=best_l1_optuna,
    fit_intercept=False,
    max_iter=10000
)
enet_final.fit(X_tr_s, y_tr)

# Collect all results
final_models = {
    "OLS":             (ols_final.coef_,       ols_final.predict(X_te_s),     "No regularization"),
    f"Ridge(α={best_ridge_alpha:.4f})":   (theta_ridge_final,    X_te_s @ theta_ridge_final, "L2"),
    f"Lasso(α={lasso_cv.alpha_:.4f})":   (lasso_final.coef_,    lasso_final.predict(X_te_s), "L1 + feature selection"),
    f"ElasticNet(α={best_alpha_optuna:.4f})": (enet_final.coef_, enet_final.predict(X_te_s), "L1+L2 (Optuna)"),
}

rows = []
for name, (coef, preds, method) in final_models.items():
    mse = mean_squared_error(y_te, preds)
    r2  = r2_score(y_te, preds)
    nnz = int(np.sum(np.abs(coef) > 1e-6))
    rows.append({"Model": name, "Test MSE": round(mse, 4),
                 "Test R²": round(r2, 4), "# Features Used": nnz, "Method": method})

comparison_df = pd.DataFrame(rows)
print("\nFinal Model Comparison Table")
print("=" * 90)
print(comparison_df.to_string(index=False))

In [ ]:
# ── Cell 32: Lasso feature selection precision & recall vs true feature set ───
# True positives: noise features correctly zeroed; signal features correctly kept

lasso_coef_final = lasso_final.coef_
lasso_selected   = set(np.where(np.abs(lasso_coef_final) > 1e-6)[0])
true_signal_set  = set(range(N_TRUE))        # features 0..9 are signal

tp = len(lasso_selected & true_signal_set)   # selected AND truly signal
fp = len(lasso_selected - true_signal_set)   # selected AND truly noise
fn = len(true_signal_set - lasso_selected)   # NOT selected BUT truly signal

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = (2 * precision * recall / (precision + recall)
             if (precision + recall) > 0 else 0.0)

print("Lasso Feature Selection (vs true signal features)")
print("=" * 50)
print(f"Selected features        : {sorted(lasso_selected)}")
print(f"True signal features     : {sorted(true_signal_set)}")
print(f"True positives  (TP)     : {tp}  (signal features correctly included)")
print(f"False positives (FP)     : {fp}  (noise features incorrectly included)")
print(f"False negatives (FN)     : {fn}  (signal features incorrectly excluded)")
print(f"Precision                : {precision:.3f}")
print(f"Recall                   : {recall:.3f}")
print(f"F1 Score                 : {f1:.3f}")

# Visual: true vs Lasso coefficients
fig, ax = plt.subplots(figsize=(12, 4))
x_pos = np.arange(N_FEATURES)
ax.bar(x_pos - 0.2, true_coef_full,       width=0.4, label="True coefficients",  color="#2ca02c", alpha=0.7)
ax.bar(x_pos + 0.2, lasso_coef_final,     width=0.4, label="Lasso recovered",    color="#d62728", alpha=0.7)
ax.axhline(0, color="black", linewidth=0.7)
ax.axvline(N_TRUE - 0.5, color="gray", linestyle="--", linewidth=1, label="Signal|Noise boundary")
ax.set_xlabel("Feature Index")
ax.set_ylabel("Coefficient")
ax.set_title("True vs Lasso-Recovered Coefficients", fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 33: Summary dashboard ────────────────────────────────────────────────
print("\n" + "="*60)
print(" LAB SUMMARY DASHBOARD")
print("="*60)

print("\n[1] FROM-SCRATCH IMPLEMENTATIONS")
print(f"    Ridge (analytical): max diff vs sklearn = {max_diff_ridge:.2e}")
print(f"    Lasso (coord desc): max diff vs sklearn = {max_diff_lasso:.4f}")

print("\n[2] OPTIMAL REGULARIZATION")
print(f"    Ridge best α  : {best_ridge_alpha:.6f}")
print(f"    Lasso best α  : {lasso_cv.alpha_:.6f}")
print(f"    ElasticNet α  : {best_alpha_optuna:.6f}, l1_ratio={best_l1_optuna:.3f}")

print("\n[3] FEATURE SELECTION (Lasso)")
print(f"    Features selected : {len(lasso_selected)} / 25")
print(f"    Precision / Recall: {precision:.3f} / {recall:.3f}")

print("\n[4] OPTIMIZER COMPARISON (final loss)")
for name, losses in [("SGD", l_sgd), ("Momentum", l_mom), ("Adam", l_adam)]:
    print(f"    {name:<12}: {losses[-1]:.4f}")

print("\n[5] HPO (50 evaluations each)")
print(f"    Optuna/GP best : {best_so_far_optuna[:N_TRIALS][-1]:.4f}")
print(f"    Grid search    : {best_grid_mse:.4f}")
print(f"    Random search  : {best_rand_mse:.4f}")
print("="*60)

## Section 7 — Self-Check: Five Conceptual Questions

Work through each question before expanding the answer.

---

**Q1.** Lasso is used with 25 features (10 real). After fitting with optimal α, 12 features are non-zero. Does this mean Lasso correctly identified all 10 real features?

<details>
<summary>Show Answer</summary>

Not necessarily. Having 12 non-zero features means Lasso selected 12 of the 25 — but the identity of those 12 matters, not just the count. The 12 could include all 10 true signal features plus 2 noise features (good recall, imperfect precision), or they could miss some true features while including noise ones. You need to compute **precision** (of the 12 selected, how many are truly signal?) and **recall** (of the 10 true signals, how many were selected?). A count of 12 alone cannot tell you whether feature selection was correct.

</details>

---

**Q2.** Ridge achieves test R²=0.88; OLS achieves test R²=0.82 on the same data. Why does Ridge beat OLS?

<details>
<summary>Show Answer</summary>

This dataset has **15 noise features** — OLS fits to all 25 features including the noise, leading to overfitting (high variance). The OLS coefficients on noise features are non-zero due to random correlations in the training set, which hurt generalization. Ridge penalises the L2 norm of all coefficients, effectively **shrinking the noise feature coefficients toward zero**. This reduces variance at the cost of a small bias increase — but since the bias is much smaller than the variance reduction, the net effect on test error is positive (bias-variance tradeoff in action).

</details>

---

**Q3.** Adam uses lr=0.001 as default. Your SGD needs lr=0.0001 to converge. What accounts for the 10× difference?

<details>
<summary>Show Answer</summary>

Adam has **built-in adaptive scaling** of the learning rate per parameter. The denominator $\sqrt{\hat{v}} + \varepsilon$ scales down the effective step by the magnitude of recent gradients. For parameters with large gradients, the effective step is automatically reduced; for parameters with small gradients, the effective step is amplified. This means the *nominal* lr=0.001 is not applied directly — it is divided by the gradient magnitude estimate. SGD's lr=0.001 applies the full step regardless of gradient scale, which can cause oscillation or divergence on steep directions. Adam's adaptive normalisation makes it robust to the absolute scale of the loss landscape, effectively behaving as if using a much smaller lr on steep directions.

</details>

---

**Q4.** Your Lasso regularization path shows Feature #7 (a noise feature) entering at α=0.05. What does this mean?

<details>
<summary>Show Answer</summary>

Feature #7's coefficient becomes non-zero when α drops below 0.05. This means that at α=0.05, the regularization is *just* weak enough that the spurious correlation of Feature #7 with the target (in this finite sample) outweighs the L1 penalty. In other words, Feature #7 has a small but non-negligible correlation with the target in the training data — purely by chance — that becomes detectable once the penalty is relaxed below 0.05. It does **not** mean Feature #7 is a real predictor. This is a false positive induced by finite-sample randomness. Increasing α above 0.05 correctly zeros it out.

</details>

---

**Q5.** Optuna found best α=0.08 and l1_ratio=0.65. Without Optuna, your grid search only checked l1_ratio=[0.0, 0.5, 1.0]. What would grid search have returned?

<details>
<summary>Show Answer</summary>

Grid search would have evaluated l1_ratio ∈ {0.0, 0.5, 1.0}. Since the true optimal l1_ratio=0.65 lies **between** 0.5 and 1.0 and is not in the grid, grid search would return the closest grid point — likely **l1_ratio=0.5** (pure Ridge direction) or **l1_ratio=1.0** (pure Lasso), whichever has lower CV MSE. Neither of these is the true optimum. The model returned by grid search would be sub-optimal, and its performance gap vs Optuna would grow the further the true optimum is from the nearest grid point. This illustrates how grid search wastes its budget on uniform coverage rather than concentrating evaluations near the promising region.

</details>